# 03 · In-silico perturbations and Mana et al. (2023) priors

Visualises `src/perturbation` output (run `python -m src.perturbation.run`). Higher map values = the perturbation
moves the subject toward the target (healthy mean FC for `p2h_*`, patient mean FC for `p2p_apos`); best node = argmax.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pickle
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from src.core import paths
from src.core.data import GROUPS, load_data, load_model_results, load_features

sns.set_theme(style="whitegrid", context="notebook")
PALETTE = {"HC": "#2a9d8f", "AP": "#e76f51", "NAP": "#e9c46a"}

In [ ]:
D = paths.DERIVED / "perturbation"
mr = load_model_results(); res, atlas = mr["results"], mr["atlas"]
labels, network = atlas["labels"], np.array(atlas["network"])
sets = pd.read_csv(D / "mana_sets.csv")
rest = sets.query("set == 'restorative'").node.values; path = sets.query("set == 'pathogenic'").node.values
display(sets)
display(pd.read_csv(D / "mana_tests.csv").round(4))

## Mean perturbation maps (red = Mana restorative, navy = Mana pathogenic)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(17, 11), sharex=True)
for ax, (mapname, who) in zip(axes, (("p2h_noise", ("AP", "NAP")), ("p2h_aneg", ("AP", "NAP")), ("p2p_apos", ("HC",)))):
    M = np.array([res[g][k][mapname] for g in who for k in res[g]])
    cols = ["crimson" if i in rest else ("navy" if i in path else "lightgray") for i in range(100)]
    ax.bar(range(100), M.mean(0), yerr=M.std(0) / np.sqrt(len(M)), color=cols)
    ax.set_title(f"{mapname} — mean ± sem over {'+'.join(who)}"); ax.axhline(0, color="k", lw=.6)
axes[-1].set_xticks(range(100)); axes[-1].set_xticklabels(labels, rotation=90, fontsize=6)
plt.tight_layout(); plt.show()

## Best node per subject

In [ ]:
B = pd.read_csv(D / "best_nodes.csv")
for mapname, who in (("p2h_noise", ("AP", "NAP")), ("p2h_aneg", ("AP", "NAP")), ("p2p_apos", ("HC",))):
    b = B[(B["map"] == mapname) & B.group.isin(who)]
    print(f"{mapname} ({'+'.join(who)}):", b.best_label.value_counts().head(6).to_dict(),
          f"| in Mana restorative: {b.in_mana_restorative.mean():.2f}, pathogenic: {b.in_mana_pathogenic.mean():.2f}")

In [ ]:
from nilearn import plotting
b = B[(B["map"] == "p2h_noise") & B.group.isin(["AP", "NAP"])]
counts = np.bincount(b.best_node, minlength=100)
sel = counts > 0
plotting.plot_markers(counts[sel], atlas["coords"][sel], node_size=20 + 8 * counts[sel], node_cmap="magma_r",
                      title="best P2H (noise) node across patients: # subjects", display_mode="lzry")
plt.show()

## Map summaries by group and vs symptoms

In [ ]:
S = pd.read_csv(D / "subject_measures.csv")
display(pd.read_csv(D / "group_tests.csv").round(4))
display(pd.read_csv(D / "symptom_correlations.csv").sort_values("p_perm").round(4))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, m in zip(axes, ("p2h_noise_max", "p2h_aneg_max", "p2h_noise_mana_rest")):
    sns.boxplot(data=S, x="group", y=m, order=GROUPS, palette=PALETTE, ax=ax, showfliers=False)
    sns.stripplot(data=S, x="group", y=m, order=GROUPS, color="k", alpha=.35, size=3, ax=ax)
plt.tight_layout(); plt.show()